[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Carlo-Broschi/statistics-python-for-students/blob/main/03_%E7%B5%B1%E8%A8%88_3%E7%B4%9A/01_%E5%88%86%E6%95%A3%E3%83%BB%E6%A8%99%E6%BA%96%E5%81%8F%E5%B7%AE%E3%83%BB%E6%A8%99%E6%BA%96%E5%8C%96.ipynb)

> 🟢 **Colab（ブラウザ）で実行できます。** Privateリポジトリは初回にColabでGitHub連携の許可が必要です。
> 最初に下の「セットアップ」セルを実行してください（Colabでは教材データを自動生成、ローカルでは何もしません）。

In [ ]:
# === ① セットアップ（最初に実行してください）===============================
# Colabなどデータが無い環境では、教材と同一の合成データを自動生成します。
# ローカル/Codespacesで data/ がある場合は何もしません（再現性のため乱数seedは固定）。
import os
if not os.path.exists('../data/students_scores.csv'):
    os.makedirs('/content/nb', exist_ok=True); os.chdir('/content/nb')  # ../ を有効化
    os.makedirs('../data', exist_ok=True)
    import numpy as np, pandas as pd
    D = '../data'; rng = np.random.default_rng(42)
    n = 120; cls = rng.choice(['A','B','C'], n)
    math = np.clip(rng.normal(65,15,n),0,100).round().astype(int)
    eng = np.clip(0.6*math+rng.normal(28,10,n),0,100).round().astype(int)
    jp = np.clip(rng.normal(70,12,n),0,100).round().astype(int)
    hrs = np.clip(rng.normal(1.5,0.8,n)+math/100,0,None).round(1)
    pd.DataFrame({'生徒ID':[f'S{i:03d}' for i in range(1,n+1)],'クラス':cls,
        '数学':math,'英語':eng,'国語':jp,'勉強時間':hrs}).to_csv(f'{D}/students_scores.csv',index=False,encoding='utf-8-sig')
    days = pd.date_range('2026-06-01', periods=30)
    temp = (24+4*np.sin(np.arange(30)/5)+rng.normal(0,2,30)).round(1)
    rain = np.clip(rng.gamma(1.2,4,30)*(rng.random(30)<0.4),0,None).round(1)
    pd.DataFrame({'日付':days.strftime('%Y-%m-%d'),'気温':temp,'降水量':rain}).to_csv(f'{D}/weather.csv',index=False,encoding='utf-8-sig')
    m = 400; inds = rng.choice(['製造','IT','小売','医療','金融','建設'],m,p=[.25,.2,.18,.12,.15,.1])
    chs = ['展示会','Web広告','メルマガ','紹介','テレアポ']; ch = rng.choice(chs,m,p=[.2,.3,.2,.15,.15])
    deal = (rng.lognormal(13,0.7,m)).round(-3).astype(int)
    wr = {'展示会':.35,'Web広告':.18,'メルマガ':.12,'紹介':.45,'テレアポ':.15}
    won = np.array([rng.random()<wr[c] for c in ch])
    dt = pd.to_datetime('2026-01-01')+pd.to_timedelta(rng.integers(0,180,m),unit='D')
    pd.DataFrame({'商談ID':[f'D{i:04d}' for i in range(1,m+1)],'受注日':dt.strftime('%Y-%m-%d'),
        '業界':inds,'獲得チャネル':ch,'商談金額':deal,'担当者':rng.choice(['田中','鈴木','佐藤','高橋'],m),
        '受注':won.astype(int)}).to_csv(f'{D}/sales_btob.csv',index=False,encoding='utf-8-sig')
    months = pd.date_range('2026-01-01',periods=6,freq='MS').strftime('%Y-%m')
    base = {'展示会':2000,'Web広告':12000,'メルマガ':6000,'紹介':800,'テレアポ':1500}
    ctr = {'展示会':.08,'Web広告':.03,'メルマガ':.05,'紹介':.2,'テレアポ':.12}
    cvr = {'展示会':.12,'Web広告':.04,'メルマガ':.06,'紹介':.25,'テレアポ':.09}
    cpc = {'展示会':30,'Web広告':80,'メルマガ':5,'紹介':0,'テレアポ':200}
    rows = []
    for mo in months:
        for c in chs:
            imp = int(base[c]*rng.uniform(0.8,1.3)); clk = int(imp*ctr[c]*rng.uniform(0.8,1.2))
            cv = int(clk*cvr[c]*rng.uniform(0.7,1.3)); cost = int(clk*cpc[c]*rng.uniform(0.9,1.1))
            rows.append([mo,c,imp,clk,cv,cost])
    pd.DataFrame(rows,columns=['月','チャネル','表示回数','クリック数','獲得数','費用']).to_csv(f'{D}/web_marketing.csv',index=False,encoding='utf-8-sig')
    ab = []
    for grp,p,size in [('A_青ボタン',0.082,2400),('B_緑ボタン',0.104,2380)]:
        for c in (rng.random(size)<p): ab.append([grp,int(c)])
    pd.DataFrame(ab,columns=['グループ','申込']).sample(frac=1,random_state=1).reset_index(drop=True).to_csv(f'{D}/ab_test.csv',index=False,encoding='utf-8-sig')
    print('✅ 合成データを生成しました（Colab環境）')
else:
    print('✅ data/ を検出しました（ローカル/Codespaces）')

# 統計3級-01. 分散・標準偏差・標準化

ばらつきを「1つの数」で表す本格的な指標を学びます。
偏差値の正体もここでわかります。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
plt.rcParams['axes.unicode_minus'] = False
df = pd.read_csv('../data/students_scores.csv')
df.head()

## 1. 偏差 → 分散 → 標準偏差

1. **偏差** = 各データ − 平均（平均からのズレ）
2. **分散** = 偏差を2乗して平均したもの
3. **標準偏差** = 分散の平方根（単位が元データと同じに戻る）

$$ 分散\ s^2 = \frac{1}{n}\sum (x_i - \bar{x})^2, \quad 標準偏差\ s = \sqrt{s^2} $$

In [ ]:
x = np.array([60, 70, 80, 90, 100])
mean = x.mean()
deviation = x - mean
print('偏差:', deviation)
print('分散:', np.mean(deviation**2))         # 手計算
print('分散(np):', x.var())                   # numpyは標本分散(÷n)
print('標準偏差:', x.std().round(2))

> ⚠️ 分散には2種類あります。`÷n`（母分散）と `÷(n-1)`（不偏分散）。
pandasの `.var()` は既定で `÷(n-1)`、numpyは `÷n`。3級では区別を意識しましょう。

In [ ]:
s = pd.Series([60, 70, 80, 90, 100])
print('pandas .var() ÷(n-1):', s.var())
print('numpy  .var() ÷n   :', np.var(s))
print('ddof指定で合わせる :', np.var(s, ddof=1))

## 2. データを比べる：標準化（z得点）

単位や平均が違うデータを公平に比べるため、
$$ z = \frac{x - 平均}{標準偏差} $$
に変換します。平均0・標準偏差1になります。

In [ ]:
math = df['数学']
z = (math - math.mean()) / math.std()
print('標準化後の平均:', round(z.mean(), 5), ' 標準偏差:', round(z.std(), 3))

## 3. 偏差値

z得点を見やすくしたものが偏差値です。
$$ 偏差値 = 50 + 10 \times z $$
平均が偏差値50、標準偏差が10になります。

In [ ]:
df['数学偏差値'] = 50 + 10 * (math - math.mean()) / math.std()
df[['生徒ID', '数学', '数学偏差値']].sort_values('数学', ascending=False).head()

---
## 🏆 練習問題

**問1.** `[2, 4, 6, 8, 10]` の分散と標準偏差を手計算の式で求めよう。

**問2.** ある生徒は数学70点、英語70点だった。`数学`と`英語`それぞれで偏差値を計算し、
「どちらの教科のほうが相対的に良かったか」を答えよう。

**問3.** `国語` の偏差値列を作り、偏差値60以上の生徒が何人いるか数えよう。

In [ ]:
# 問1


In [ ]:
# 問2


In [ ]:
# 問3


<details><summary>解答例</summary>

```python
x = np.array([2,4,6,8,10]); print(np.var(x), np.std(x))  # 8, 2.83
for col in ['数学','英語']:
    m, s = df[col].mean(), df[col].std()
    print(col, 50 + 10*(70-m)/s)
kokugo_hensa = 50 + 10*(df['国語']-df['国語'].mean())/df['国語'].std()
print((kokugo_hensa >= 60).sum())
```
</details>